# Homework 04 - TinyValue Implementation

目标：不是读懂 micrograd，而是亲手造一个最小自动求导类。顺序是：前向值 -> 局部反传 -> 拓扑排序 -> 整图 backward。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check

## 完整例子 - 先只看前向对象

这个小对象不会自动求导，只让你确认：`data` 是值，`grad` 先是 0。

In [ ]:
class Box:
    def __init__(self, data):
        self.data = data
        self.grad = 0

x = Box(2.0)
print('x.data =', x.data)
print('x.grad =', x.grad)

## Modify - 先只补加法反传这一行

在完整实现之前，先单独练一行：加法的两个输入都收到 `out.grad`。

In [ ]:
# 填写说明：
# - 完成 student_add_backward_rule，模拟加法反传给两个父节点的贡献。
# - 运行本 cell，看 qa_check 的提示。

def student_add_backward_rule(out_grad):
    # TODO: 返回 self 和 other 应该各自累加多少 grad。
    pass


qa_check('qa_check_04_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- 加法节点不会改变上游梯度，`out.grad` 原样分给左右两个父节点。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_add_backward_rule(out_grad):
    return out_grad, out_grad
```

</details>


## 作业 1-6 - 填 MyTinyValue

先让加法/乘法前向能跑，再补 `_backward`。不要一次想完整个类，每次只修一个测试。

In [ ]:
class MyTinyValue:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, MyTinyValue) else MyTinyValue(other)
        out = MyTinyValue(self.data + other.data, (self, other), '+')

        def _backward():
            # TODO 1: 加法反传，两个输入都收到 out.grad
            pass

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, MyTinyValue) else MyTinyValue(other)
        out = MyTinyValue(self.data * other.data, (self, other), '*')

        def _backward():
            # TODO 2: 乘法反传：self 收 other.data*out.grad，other 收 self.data*out.grad
            pass

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = MyTinyValue(self.data ** other, (self,), f'**{other}')

        def _backward():
            # TODO 3: x**n 的导数是 n*x**(n-1)，还要乘 out.grad
            pass

        out._backward = _backward
        return out

    def relu(self):
        out = MyTinyValue(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            # TODO 4: out.data > 0 时传 out.grad，否则传 0
            pass

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            # TODO 5: 如果访问过就返回；否则递归父节点，最后 append 自己
            pass

        build_topo(self)
        # TODO 6: 最终节点自己对自己导数是 1，然后反向执行 _backward
        pass

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __repr__(self):
        return f'MyTinyValue(data={self.data}, grad={self.grad})'

In [ ]:
# 填写说明：
# - 不需要额外填变量；先完成上方 MyTinyValue 的前向建图，再运行检查。
# - 填完后运行本 cell，看 qa_check 的提示。

qa_check('qa_check_04_forward', globals(), MyTinyValue)

<details><summary>Show / Hide 本题提示</summary>

- 先只看一个运算节点：forward 创建 `out`，backward 用 `out.grad` 给父节点累加；不要忘记拓扑排序后反向执行。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `c.data` 填 `5.0`。
- `d.data` 填 `6.0`。
- `c._op` 填 `'+'`。
- `d._op` 填 `'*'`。

</details>

In [ ]:
# 填写说明：
# - 不需要额外填变量；先完成 MyTinyValue 的 backward/pow/relu，再运行检查。
# - 填完后运行本 cell，看 qa_check 的提示。

qa_check('qa_check_04_backward', globals(), MyTinyValue)

<details><summary>Show / Hide 本题提示</summary>

- 先只看一个运算节点：forward 创建 `out`，backward 用 `out.grad` 给父节点累加；不要忘记拓扑排序后反向执行。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `x.grad` 约等于 `6.0`。
- `out.data` 填 `0 and r.grad == 0`。
- `p.grad` 约等于 `6.0`。

</details>

<details><summary>Show / Hide 提示</summary>

`build_topo` 的形状是：访问自己 -> 递归父节点 -> append 自己。`backward` 的形状是：`self.grad = 1`，然后 `for node in reversed(topo): node._backward()`。

</details>

## Debug Lab - 修真实错误

In [ ]:
# 填写说明：
# - 完成 student_pow_relu_debug_values，用数字证明 pow 要乘 out.grad，ReLU 的值不是 1。
# - 运行本 cell，看 qa_check 的提示。

def student_pow_relu_debug_values():
    # Debug 1: x=3, n=2, out.grad=5 时，pow 传给 x 的 grad 是多少？
    pow_grad = None
    # Debug 2: ReLU(3) 的前向值是多少？
    relu_value = None
    # Debug 3: ReLU 在 x=3 处的局部导数是多少？
    relu_grad = None
    return pow_grad, relu_value, relu_grad


qa_check('qa_check_04_debug', globals())


<details><summary>Show / Hide 本题提示</summary>

- `x**n` 的局部导数是 `n * x ** (n-1)`，反传还要乘上游的 `out.grad`。
- `ReLU(3)` 的值是 3；导数才是 1。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_pow_relu_debug_values():
    pow_grad = 2 * 3 ** (2 - 1) * 5
    relu_value = 3
    relu_grad = 1
    return pow_grad, relu_value, relu_grad
```

</details>


## 提交清单

- [ ] `qa_check_04_forward` 通过
- [ ] `qa_check_04_backward` 通过
- [ ] Debug Lab 通过
- [ ] 我能解释为什么 `+=`、`out.grad`、拓扑排序缺一不可

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>